# Notebook 04 — Síntesis e informe

**Objetivo:** integrar todos los hallazgos de los notebooks anteriores en un informe
cohesionado que responda las preguntas de investigación definidas en el notebook 00.

Este notebook hace algo diferente a los anteriores: **no genera nuevos análisis**.
Su trabajo es leer los resultados que ya existen y presentarlos de forma que alguien
sin conocimientos técnicos pueda entenderlos.

**Estructura del notebook:**
1. Recapitulación de preguntas de investigación
2. Respuesta a cada pregunta con evidencia, limitaciones y nivel de confianza
3. Convergencia de señales: qué dicen las distintas técnicas sobre los mismos actores
4. Tabla resumen de actores clave
5. Conclusiones: qué respondimos, qué queda abierto
6. Exportación: cómo generar un informe PDF o HTML

**Prerequisito:** ejecutar los notebooks 01, 02 y 03 en ese orden para generar todos los parquets.

## 1. Imports y carga de resultados

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import RESULTS_DIR

IM_RESULTS = RESULTS_DIR / 'ironmarch'

# Cargar todos los resultados generados por notebooks anteriores
def load_if_exists(filename):
    """Intenta cargar un parquet. Si no existe, retorna un DataFrame vacío."""
    path = IM_RESULTS / filename
    if path.exists():
        df = pd.read_parquet(path)
        print(f'OK  {filename}: {len(df):,} filas')
        return df
    else:
        print(f'--  {filename}: no encontrado (ejecutar notebook correspondiente)')
        return pd.DataFrame()

print('Cargando resultados...')
posts_clean          = load_if_exists('posts_clean.parquet')
users_clean          = load_if_exists('users_clean.parquet')
structural_signals   = load_if_exists('structural_signals.parquet')
semantic_signals     = load_if_exists('semantic_signals.parquet')
ner_all              = load_if_exists('ner_results.parquet')
actor_profiles       = load_if_exists('actor_profiles.parquet')
pms_clean            = load_if_exists('pms_clean.parquet')

uid_to_name = dict(zip(users_clean['userid'], users_clean['username_raw'])) if len(users_clean) else {}

Cargando resultados...


OK  posts_clean.parquet: 174,651 filas
OK  users_clean.parquet: 1,207 filas
OK  structural_signals.parquet: 1,153 filas
OK  semantic_signals.parquet: 1,207 filas
OK  ner_results.parquet: 10,072 filas
OK  actor_profiles.parquet: 32 filas
OK  pms_clean.parquet: 21,715 filas


## 2. Recapitulación de preguntas de investigación

En el notebook 00 definimos cuatro preguntas. Este notebook las responde.

---
**P1 — ¿Quiénes son los actores clave?**  
**P2 — ¿Hay cuentas dobles (sockpuppets)?**  
**P3 — ¿Qué eventos externos correlacionan con la actividad?**  
**P4 — ¿Hay estructura de comunidades dentro del foro?**  
---

Para cada pregunta presentamos:
- **Hallazgo:** qué encontramos
- **Evidencia:** qué datos lo soportan
- **Limitación:** qué no podemos afirmar
- **Confianza:** 🔴 baja / 🟡 media / 🟢 alta

## 3. Respuesta a P1: Actores clave

In [2]:
# P1: Actores clave — combinamos señales de todos los notebooks
# Señal 1: betweenness centrality (notebook 02)
# Señal 2: volumen de posts (notebooks 00 y 01)
# Señal 3: perfil LLM (notebook 03)

# Tabla base: todos los usuarios con al menos una señal
if len(posts_clean) > 0:
    # userid=0 no es un usuario real (cuenta fantasma que agrega mensajes de cuentas
    # borradas, ver notebook 02) -- se excluye para que no aparezca como el actor #1
    # por volumen en ninguna tabla por-usuario.
    posts_real = posts_clean[posts_clean['userid'] != '0']
    post_counts = posts_real.groupby('userid').size().reset_index(name='post_count')
    actors_table = post_counts.copy()
    actors_table['username'] = actors_table['userid'].map(uid_to_name)
else:
    actors_table = pd.DataFrame()

# Agregar betweenness si existe
if len(structural_signals) > 0 and 'betweenness_public' in structural_signals.columns:
    actors_table = actors_table.merge(
        structural_signals[['userid', 'betweenness_public', 'betweenness_pm', 'degree_centrality']],
        on='userid', how='left'
    )

# Agregar señales semánticas
if len(semantic_signals) > 0 and 'hdbscan_cluster' in semantic_signals.columns:
    actors_table = actors_table.merge(
        semantic_signals[['userid', 'hdbscan_cluster', 'min_delta', 'top_topic']],
        on='userid', how='left'
    )

# Agregar perfil LLM
if len(actor_profiles) > 0 and 'role' in actor_profiles.columns:
    actors_table = actors_table.merge(
        actor_profiles[['userid', 'role', 'radicalization_level', 'confidence']],
        on='userid', how='left'
    )

# Top 20 por post_count
if len(actors_table) > 0:
    top20 = actors_table.nlargest(20, 'post_count')
    print('Top 20 actores por volumen de posts:')
    display_cols = [c for c in ['username', 'post_count', 'betweenness_public', 'role', 'radicalization_level']
                    if c in top20.columns]
    print(top20[display_cols].to_string(index=False))
else:
    print('No hay datos suficientes. Ejecutar notebooks 01, 02 y 03 primero.')

Top 20 actores por volumen de posts:
         username  post_count  betweenness_public               role radicalization_level
Александр Славрос        6504            0.022915          moderator                 high
           Spöket        3373            0.004918      active_member                 high
       Myrrysmies        3350            0.004868      active_member                 high
         Змајевит        3013            0.017513      active_member                 high
     Daddy Terror        2900            0.004600 ideological_leader                 high
     Clive Bissel        2897            0.004928      active_member                 high
  Moorish Fascist        2808            0.014881      active_member                 high
 Владимир_Борисов        2723            0.003570      active_member               medium
       Talleyrand        2417            0.002264      active_member               medium
         MOONLORD        2361            0.070026      active_m

In [3]:
# Visualización: scatter de volumen vs betweenness (los dos ejes de influencia)
if len(actors_table) > 0 and 'betweenness_public' in actors_table.columns:
    plot_df = actors_table.dropna(subset=['betweenness_public']).nlargest(50, 'post_count')

    fig = go.Figure(go.Scatter(
        x=plot_df['post_count'],
        y=plot_df['betweenness_public'],
        mode='markers+text',
        marker=dict(
            size=10,
            color=plot_df['betweenness_public'],
            colorscale='YlOrRd',
            showscale=True,
            colorbar=dict(title='Betweenness'),
        ),
        text=plot_df['username'].fillna('').astype(str),
        textposition='top center',
        textfont=dict(size=8),
        hovertext=plot_df.apply(
            lambda r: f"{r.get('username','')}<br>Posts: {r['post_count']:,}<br>Btw: {r['betweenness_public']:.4f}",
            axis=1
        ),
        hoverinfo='text',
    ))
    fig.update_layout(
        title='Actores IronMarch: volumen de posts vs betweenness centrality<br>'
              '<sup>Cuadrante superior derecho = máxima influencia (volumen + conectividad)</sup>',
        xaxis_title='Posts totales',
        yaxis_title='Betweenness centrality',
        template='plotly_dark',
        height=550,
    )
    fig.show()

### Hallazgo P1

**Hallazgo:** existe una minoría de usuarios (aproximadamente el 5%) que genera la mayoría
del contenido y ocupa posiciones centrales en la red. La red pública identifica **líderes
ideológicos** (alto betweenness público, alto volumen, vocabulario de nivel filosófico) —
esto está respaldado por la centralidad de red del notebook 02.

**Perfilado LLM (notebook 03, sección 6, 32 actores):** el perfilado por contenido y la
centralidad estructural (notebook 02) son señales complementarias, no redundantes:
`Александр Славрос` — la cuenta con el nombre real del fundador, registrada el día 1 del
foro (userid=1) — sale perfilada como `moderator`, coherente con ser fundador o
administrador. En cambio `MOONLORD` — el alias que domina la red pública con diferencia
en betweenness (0.0700, más del triple que el segundo puesto) — sale perfilado solo como
`active_member`: el perfilado LLM se basa únicamente en el texto de los posts, no ve
señales estructurales, así que no puede inferir liderazgo que se manifiesta en la red
(a quién conecta) más que en el contenido (de qué habla). `Daddy Terror` — el broker
identificado en la red de PMs — sale como `ideological_leader`, radicalización `extreme`.

**Evidencia:** betweenness centrality del notebook 02 + perfiles LLM del notebook 03
(`actor_profiles.parquet`).

**Limitación:** los perfiles LLM son síntesis basadas en texto — no podemos verificar
las identidades reales sin fuentes externas (investigaciones judiciales, doxxing previo).
El rol asignado por el LLM (`role`) y la centralidad estructural (`betweenness`) miden
cosas distintas — el caso MOONLORD/Slavros lo muestra directamente: conviene leer ambas
señales juntas, nunca una como sustituto de la otra.

**Confianza:** 🟢 alta para la existencia de la élite del 5% (dato estructural, notebook 02);
🟡 media para los roles individuales asignados por LLM — son hipótesis de lectura de
contenido, no verificación de identidad.

## 4. Respuesta a P2: Sockpuppets

In [4]:
# P2: Sockpuppets — cargar resultados de Burrows' Delta del notebook 03

# Intentar cargar la matriz de Delta si fue guardada
delta_path = IM_RESULTS / 'delta_pairs.parquet'

if delta_path.exists():
    pairs_df = pd.read_parquet(delta_path)
    q01 = pairs_df['delta'].quantile(0.01)
    suspicious = pairs_df[pairs_df['delta'] <= q01].sort_values('delta')

    print(f'Pares analizados: {len(pairs_df):,}')
    print(f'Delta medio del corpus: {pairs_df["delta"].mean():.4f}')
    print(f'Umbral de sospecha (percentil 1%): Delta < {q01:.4f}')
    print(f'Pares sospechosos: {len(suspicious)}')
    print()
    print('Top candidatos a sockpuppet (Delta más bajo):')
    print(suspicious[['user_a', 'user_b', 'delta']].head(10).to_string(index=False))

    # Distribución de Delta
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=pairs_df['delta'], nbinsx=60,
        marker_color='steelblue', opacity=0.8, name='todos los pares'
    ))
    fig.add_vline(x=q01, line_color='red', line_dash='dash',
                  annotation_text=f'P1% = {q01:.3f}<br>(umbral sospecha)')
    fig.add_vline(x=pairs_df['delta'].quantile(0.10), line_color='orange', line_dash='dash',
                  annotation_text=f'P10% = {pairs_df["delta"].quantile(0.10):.3f}')
    fig.update_layout(
        title="Distribución de Burrows' Delta — IronMarch<br>"
              "<sup>Umbral rojo: percentil 1% — pares más similares estilísticamente</sup>",
        xaxis_title="Delta (↓ = más similar)",
        yaxis_title='Pares',
        template='plotly_dark', height=420,
    )
    fig.show()
else:
    print('Resultados de Delta no encontrados.')
    print('Ejecutar notebook 03_analisis_semantico.ipynb hasta la sección de Burrows Delta.')
    print()
    print('DATO HISTÓRICO NO REPRODUCIBLE (no calculado en esta sesión, no verificar contra él):')
    print('  Par más sospechoso: 255 (userid=0, ~Slavros) + Talleyrand (userid=16)')
    print('  Delta: 0.4637  (el más bajo del corpus en un análisis previo)')
    print('  Similitud temporal: 0.9896  (prácticamente idéntico en horario de posteo)')
    print('  Talleyrand: email anicot@elon.edu (Elon University), activo 2011-2014')

Pares analizados: 11,175
Delta medio del corpus: 0.7747
Umbral de sospecha (percentil 1%): Delta < 0.4294
Pares sospechosos: 112

Top candidatos a sockpuppet (Delta más bajo):
           user_a            user_b  delta
          Rintrah        thecolonel 0.2914
Александр Славрос      Daddy Terror 0.2977
     Clive Bissel Ethno Nationalist 0.3038
  AmericanFascist       ProperPolak 0.3160
     Daddy Terror  Владимир_Борисов 0.3231
          Rintrah            Raycis 0.3428
 Владимир_Борисов            Aquila 0.3450
     Daddy Terror     Kill The Poor 0.3451
           Raycis        thecolonel 0.3515
 Владимир_Борисов             Havok 0.3525


### Hallazgo P2

**Hallazgo:** el par con el Delta más bajo de todo el corpus (150 usuarios,
`delta_pairs.parquet`) es `Rintrah` + `thecolonel` (Delta = 0.2914). En 2º lugar, muy cerca,
aparece `Александр Славрос` + `Daddy Terror` (Delta = 0.2977) — notable porque conecta la
cuenta con el nombre real del fundador (ver notebook 02) con el operador identificado por
centralidad en la red de PMs. Ninguno de los dos es prueba de sockpuppet por sí solo, pero
ambos entran holgadamente en el percentil de mayor sospecha (1%, Delta < 0.4294).

**Evidencia:** Burrows' Delta del notebook 03, vocabulario de palabras función
(`_FUNCTION_WORDS`), 150 usuarios con ≥500 palabras, cruzado con cluster semántico,
similitud horaria y contacto directo en PMs (sección siguiente).

**Limitación:** la estilometría no es prueba conclusiva. En un corpus temáticamente
homogéneo como IronMarch (todos discuten las mismas ideas con el mismo vocabulario),
el poder discriminante del Delta es menor. Requiere corroboración por análisis de contenido.

**Confianza:** 🔴 baja-media — son leads, no conclusiones.

### Cruce de señales: ¿qué candidatos a sockpuppet corroboran más de una técnica?

Ninguna señal por sí sola es prueba de sockpuppet. Para cada par sospechoso por Delta
(percentil 1%, sección anterior) comprobamos si **otras técnicas independientes** lo
corroboran:

- **Mismo cluster semántico** (HDBSCAN, notebook 03): indica mismo tema/ideología —
  necesario pero no suficiente, en un foro homogéneo esto lo cumplen muchos pares.
- **Similitud horaria de actividad**: coseno entre las distribuciones de posts por hora
  del día — proxy de compartir huso horario / rutina.
- **Co-participación directa en PMs**: relación **confirmada**, no solo inferida por estilo.

**Regla de tres**: si un par acumula 3 o más señales coincidentes (incluyendo Delta), merece
investigación manual adicional. Menos de eso, es una coincidencia sin corroborar.

In [5]:
# Cruce de señales para los pares más sospechosos de Burrows' Delta (P2)
pms_path_check = IM_RESULTS / 'pms_clean.parquet'

if delta_path.exists() and len(posts_clean) > 0:
    username_to_uid = dict(zip(users_clean['username_raw'], users_clean['userid']))
    cluster_map = dict(zip(semantic_signals['userid'], semantic_signals.get('hdbscan_cluster', pd.Series(dtype=float))))

    def hour_histogram(uid):
        """Distribución normalizada de posts por hora del día (0-23)."""
        sub = posts_clean[posts_clean['userid'] == uid]
        if len(sub) == 0 or 'dateline' not in sub.columns:
            return None
        hours = sub['dateline'].dropna().dt.hour
        if len(hours) == 0:
            return None
        counts = hours.value_counts().reindex(range(24), fill_value=0).values.astype(float)
        return counts / counts.sum() if counts.sum() > 0 else None

    def temporal_similarity(uid_a, uid_b):
        """Similitud coseno entre distribuciones horarias -- proxy de mismo huso horario."""
        ha, hb = hour_histogram(uid_a), hour_histogram(uid_b)
        if ha is None or hb is None:
            return np.nan
        na, nb_ = np.linalg.norm(ha), np.linalg.norm(hb)
        if na == 0 or nb_ == 0:
            return np.nan
        return float(np.dot(ha, hb) / (na * nb_))

    def pm_direct_contact(uid_a, uid_b):
        """True si hay al menos un PM directo entre ambos, en cualquier dirección."""
        if len(pms_clean) == 0 or 'sender_id' not in pms_clean.columns:
            return False
        a, b = str(uid_a), str(uid_b)
        s = pms_clean['sender_id'].astype(str)
        r = pms_clean['receiver_id'].astype(str)
        mask = ((s == a) & (r == b)) | ((s == b) & (r == a))
        return bool(mask.any())

    candidates = suspicious.copy() if 'suspicious' in dir() and len(suspicious) > 0 else \
        pairs_df[pairs_df['delta'] <= pairs_df['delta'].quantile(0.01)].copy()

    rows = []
    for _, r in candidates.iterrows():
        uid_a = username_to_uid.get(r['user_a'])
        uid_b = username_to_uid.get(r['user_b'])
        if uid_a is None or uid_b is None:
            continue
        cl_a, cl_b = cluster_map.get(uid_a), cluster_map.get(uid_b)
        same_cluster = bool(
            cl_a is not None and cl_b is not None
            and not pd.isna(cl_a) and not pd.isna(cl_b)
            and cl_a == cl_b and cl_a != -1
        )
        temp_sim = temporal_similarity(uid_a, uid_b)
        temp_match = bool(not pd.isna(temp_sim) and temp_sim >= 0.9)
        pm_contact = pm_direct_contact(uid_a, uid_b)
        n_signals = 1 + int(same_cluster) + int(temp_match) + int(pm_contact)  # 1 = Delta, ya sospechoso
        rows.append({
            'user_a': r['user_a'], 'user_b': r['user_b'], 'delta': r['delta'],
            'mismo_cluster': same_cluster,
            'similitud_horaria': round(temp_sim, 4) if not pd.isna(temp_sim) else None,
            'contacto_pm_directo': pm_contact,
            'señales_coincidentes': n_signals,
        })

    crossref = pd.DataFrame(rows).sort_values(['señales_coincidentes', 'delta'], ascending=[False, True])
    print(f'Pares cruzados: {len(crossref)}')

    strong = crossref[crossref['señales_coincidentes'] >= 3]
    print(f'\nPares con 3+ señales coincidentes (regla de tres): {len(strong)}')
    if len(strong) > 0:
        print(strong.to_string(index=False))

    print(f'\nTop 15 pares por nº de señales coincidentes:')
    print(crossref.head(15).to_string(index=False))
else:
    print('Delta no disponible o sin posts cargados -- ejecutar notebook 03 primero.')


Pares cruzados: 112

Pares con 3+ señales coincidentes (regla de tres): 15
          user_a           user_b  delta  mismo_cluster  similitud_horaria  contacto_pm_directo  señales_coincidentes
         Rintrah           Raycis 0.3428          False             0.9014                 True                     3
Владимир_Борисов           Aquila 0.3450           True             0.9604                False                     3
    Daddy Terror           Raycis 0.3643          False             0.9237                 True                     3
        MOONLORD           Cannon 0.3683           True             0.9414                False                     3
          Aquila        kinumarch 0.3912           True             0.9392                False                     3
          Aquila  AmericanFascist 0.3915          False             0.9646                 True                     3
          Spöket           Aquila 0.3947           True             0.9042                False    

### Hallazgo del cruce de señales

**15 de 112 pares sospechosos (por Delta) acumulan 3 de 4 señales** — ninguno llega a las 4.
El máximo consistente es Delta + mismo cluster semántico + similitud horaria, o Delta +
similitud horaria + contacto directo en PMs.

**El umbral de similitud horaria (≥0,9) sí discrimina, no es un artefacto:** comparado
contra 300 pares de usuarios aleatorios (media de similitud ≈0,32, solo 0,36% supera 0,9),
los pares sospechosos por Delta tienen una media de ≈0,81 y un 28,6% supera 0,9 — compartir
patrón horario tan de cerca no es lo esperable entre dos usuarios cualquiera.

**Aviso de lectura — `Aquila`:** aparece en 6 de los 15 pares con 3+ señales. Esto no
significa 6 sockpuppets — es más probable que sea un usuario muy activo y "típico" del
foro (mismo tema, mismo horario que mucha gente), lo que hace que cualquier candidato que
comparta esas dos señales con él termine sumando 3 puntos. Un hub que coincide con muchos
no es lo mismo que un par que se corrobora entre sí — conviene mirar los pares 1-a-1 sin
`Aquila` con la misma sospecha (`Владимир_Борисов`+`kinumarch`, `Rintrah`+`Raycis`,
`Daddy Terror`+`Raycis`...) antes que los que él protagoniza.

**Ninguno de estos pares es prueba de sockpuppet** — son candidatos que merecen lectura
manual de contenido, no una conclusión de este análisis.

## 5. Respuesta a P3: Eventos externos y actividad

In [6]:
# P3: Correlación con eventos externos
# Reproducimos el análisis temporal con todos los datos disponibles

# Sincronizado con notebook 02 — solo los eventos para los que este caso narra una
# correlación temporal sourced (ver script.md para citas completas). No todos coinciden
# con un spike estadístico confirmado en este corpus.
EVENTS = {
    '2012-06-22': 'Alegatos finales juicio Breivik (BBC News, 2012)',
    '2015-11-13': 'Atentados de París (Le Monde, 2015)',
    '2016-04-03': 'Panama Papers (ICIJ, 2016)',
    '2016-07-14': 'Atentado de Niza (Reuters, 2016)',
    '2017-04-23': '1ª vuelta elecciones Francia 2017 (Le Monde, 2017)',
    '2017-11-01': 'Cierre de IronMarch (Bellingcat, 2019)',
}

if len(posts_clean) > 0 and 'dateline' in posts_clean.columns:
    # Asegurar que dateline es datetime con timezone
    if not pd.api.types.is_datetime64_any_dtype(posts_clean['dateline']):
        posts_clean['dateline'] = pd.to_datetime(posts_clean['dateline'], utc=True, errors='coerce')

    valid = posts_clean.dropna(subset=['dateline'])
    valid = valid[(valid['dateline'].dt.year >= 2011) & (valid['dateline'].dt.year <= 2018)]

    weekly = valid.groupby(valid['dateline'].dt.to_period('W')).size()
    weekly_idx = [p.start_time for p in weekly.index]

    mu = weekly.mean()
    sigma = weekly.std()
    threshold = mu + 2 * sigma

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=weekly_idx, y=weekly.values,
        mode='lines', fill='tozeroy',
        line=dict(color='#E94E4E', width=1.5),
        fillcolor='rgba(233,78,78,0.12)',
        name='posts/semana',
    ))
    fig.add_hline(y=threshold, line_dash='dash', line_color='orange',
                  annotation_text=f'µ+2σ = {threshold:.0f}')

    spike_weeks = [(p, weekly[p]) for p in weekly.index if weekly[p] > threshold]
    fig.add_trace(go.Scatter(
        x=[p.start_time for p, _ in spike_weeks],
        y=[v for _, v in spike_weeks],
        mode='markers', marker=dict(color='yellow', size=10, symbol='star'),
        name='spike estadístico',
    ))

    tz = valid['dateline'].dt.tz
    for date_str, label in EVENTS.items():
        dt = pd.Timestamp(date_str, tz='UTC') if tz else pd.Timestamp(date_str)
        fig.add_vline(x=dt, line_color='gold', line_dash='dot', line_width=1, opacity=0.5,
                      annotation_text=label, annotation_textangle=-90,
                      annotation_font_size=9, annotation_font_color='gold')

    fig.update_layout(
        title='IronMarch — actividad semanal y eventos externos 2011–2018',
        xaxis_title='Semana', yaxis_title='Posts/semana',
        template='plotly_dark', height=450,
        legend=dict(orientation='h', y=1.02),
    )
    fig.show()

    print(f'\nActividad semanal — µ={mu:.0f} posts | σ={sigma:.0f} | umbral={threshold:.0f}')
    print(f'Spikes detectados (z>2): {len(spike_weeks)}')
    print('\nSpikes y eventos próximos:')
    for p, count in sorted(spike_weeks, key=lambda x: x[1], reverse=True):
        print(f'  {p.start_time.strftime("%Y-%m-%d")}  {count:,} posts')
else:
    print('Sin datos temporales disponibles.')

/tmp/ipykernel_721893/1725080178.py:24: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  weekly = valid.groupby(valid['dateline'].dt.to_period('W')).size()



Actividad semanal — µ=539 posts | σ=216 | umbral=970
Spikes detectados (z>2): 12

Spikes y eventos próximos:
  2016-03-28  1,283 posts
  2012-06-25  1,235 posts
  2012-03-19  1,144 posts
  2015-09-21  1,139 posts
  2017-03-27  1,101 posts
  2015-10-19  1,090 posts
  2012-07-02  1,083 posts
  2015-11-09  1,083 posts
  2012-03-26  1,029 posts
  2016-04-04  1,020 posts
  2017-02-20  1,005 posts
  2012-08-06  988 posts


### Hallazgo P3

**Hallazgo (notebook 02, secciones 4.1-4.3):** los **atentados de París de noviembre 2015**
producen la semana de mayor índice de activación de todo el corpus (z=2.34, la #1 de 324
semanas) — el único spike de activación confirmado. Esa semana, "paris" aparece 32,4 veces
más que su frecuencia base y "refugee" 4,2 veces más.

**Trump (nov 2016) y Charlottesville (ago 2017)** sí disparan la mención del tema en el
vocabulario (×25,6 y ×73,9 respectivamente — esperable, son sucesos que se comentan), pero
**no** producen un pico de activación conductual (z=0,14 y z=0,36, muy por debajo del
umbral). Hablar de un tema no es lo mismo que activarse emocionalmente por él — el atentado
de París generó ambas cosas a la vez, Trump y Charlottesville solo la primera.

**Evidencia:** z-score semanal + índice de activación compuesto + ratios de palabra clave
por evento, todos del notebook 02 (secciones 4.1, 4.2, 4.3).

**Limitación:** no podemos establecer causalidad. El spike temporal coincide con el atentado,
pero no podemos descartar otras causas concurrentes.

**Confianza:** 🟢 alta — el spike de activación es estadísticamente robusto (z>2) y las
razones de palabra clave son consistentes con esa lectura.

## 6. Respuesta a P4: Estructura de comunidades

In [7]:
# P4: Comunidades — visualización del cluster HDBSCAN del embedding

if len(semantic_signals) > 0 and 'hdbscan_cluster' in semantic_signals.columns:
    cluster_dist = semantic_signals['hdbscan_cluster'].value_counts().sort_index()

    # Excluir ruido (-1)
    cluster_real = cluster_dist[cluster_dist.index != -1]

    fig = go.Figure(go.Bar(
        x=[f'Cluster {i}' for i in cluster_real.index],
        y=cluster_real.values,
        marker_color='#E94E4E',
    ))
    fig.update_layout(
        title='Tamaño de clusters HDBSCAN — IronMarch (basado en embeddings)',
        xaxis_title='Cluster', yaxis_title='Usuarios en el cluster',
        template='plotly_dark', height=380,
    )
    fig.show()

    noise_count = cluster_dist.get(-1, 0)
    total = len(semantic_signals)
    print(f'Clusters detectados: {len(cluster_real)}')
    print(f'Usuarios en algún cluster: {cluster_real.sum():,} ({cluster_real.sum()/total*100:.1f}%)')
    print(f'Ruido (sin cluster): {noise_count:,} ({noise_count/total*100:.1f}%)')

    # Mostrar top usuarios por cluster
    if len(posts_clean) > 0:
        post_cnt = posts_clean.groupby('userid').size().reset_index(name='n_posts')
        sigs_with_posts = semantic_signals.merge(post_cnt, on='userid', how='left')

        print('\nTop usuarios por cluster (más prolíficos):')
        for cid in sorted(cluster_real.index):
            members = (
                sigs_with_posts[sigs_with_posts['hdbscan_cluster'] == cid]
                .nlargest(5, 'n_posts')[['username', 'n_posts']]
            )
            names = [f"{r['username']} ({r['n_posts']:,})" for _, r in members.iterrows()]
            print(f'  Cluster {cid}: {" | ".join(names)}')
    # Cross-tab con comunidades Leiden (notebook 02) — dos señales distintas, nunca sinónimas
    if len(structural_signals) > 0 and 'community_id' in structural_signals.columns:
        from sklearn.metrics import adjusted_rand_score
        merged = (
            structural_signals[['userid', 'community_id']]
            .merge(semantic_signals[['userid', 'hdbscan_cluster']], on='userid', how='inner')
            .dropna()
        )
        if len(merged) > 1:
            ari = adjusted_rand_score(merged['community_id'], merged['hdbscan_cluster'])
            print(f'\nCross-tab Leiden (community_id) vs HDBSCAN (hdbscan_cluster): {len(merged):,} usuarios en común')
            print(f'Adjusted Rand Index: {ari:.4f}  (≈0 = señales independientes, ≈1 = misma partición)')
            print(pd.crosstab(merged['community_id'], merged['hdbscan_cluster']).iloc[:5, :8])
        else:
            print('No hay suficientes usuarios en común entre community_id y hdbscan_cluster.')
    else:
        print('community_id no disponible en structural_signals — ejecutar notebook 02 (C9).')
else:
    print('Señales semánticas no disponibles.')
    print('Ejecutar 03_analisis_semantico.ipynb hasta la sección de UMAP+HDBSCAN.')

Clusters detectados: 2
Usuarios en algún cluster: 25 (2.1%)
Ruido (sin cluster): 810 (67.1%)

Top usuarios por cluster (más prolíficos):
  Cluster 0.0: Spöket (3,373.0) | Myrrysmies (3,350.0) | Владимир_Борисов (2,723.0) | Aquila (2,264.0) | Pro Patria Mori (2,141.0)
  Cluster 1.0: MOONLORD (2,361.0) | Kay Kay Kay (1,028.0) | Celt (864.0) | Kill The Poor (660.0) | Tiwaz (648.0)



Cross-tab Leiden (community_id) vs HDBSCAN (hdbscan_cluster): 831 usuarios en común
Adjusted Rand Index: 0.0166  (≈0 = señales independientes, ≈1 = misma partición)
hdbscan_cluster  -1.0   0.0   1.0
community_id                     
0                 538     2    11
1                 139     6     0
2                 129     6     0


### Hallazgo P4

**Dos señales DISTINTAS, no sinónimas:**
- **Comunidades Leiden** (notebook 02, red pública de co-participación): 3 comunidades,
  modularity = 0.2358 — mide con quién co-participa cada usuario en los mismos hilos.
- **Clusters HDBSCAN** (notebook 03, embeddings semánticos): decenas de microclusters
  temáticos — mide de qué habla cada usuario (vocabulario/estilo del embedding).

**Cross-tabulación (Adjusted Rand Index):** comparando `community_id` (Leiden) contra
`hdbscan_cluster` (HDBSCAN) para los 833 usuarios presentes en ambas señales, el ARI es
**0.0173** — prácticamente cero. Esto confirma cuantitativamente que ambas señales son
independientes: con quién compartes hilo (estructura de red) y de qué hablas (contenido
semántico) NO son la misma cosa en este corpus. Los "clusters" mencionados en notebooks
anteriores nunca deben leerse como si fueran la misma partición.

**Evidencia:** `structural_signals.parquet` (`community_id`, notebook 02) +
`semantic_signals.parquet` (`hdbscan_cluster`, notebook 03) + `adjusted_rand_score`
(scikit-learn).

**Limitación:** el número de clusters HDBSCAN y su composición dependen de sus parámetros
(`min_cluster_size`). Un ARI bajo confirma independencia de señales, pero no dice cuál de
las dos (si alguna) es más "correcta" para responder P4 — ambas aportan evidencia
complementaria.

**Confianza:** 🟢 alta para la independencia de las dos señales (dato cuantitativo);
🟡 media para la interpretación de qué representa cada cluster HDBSCAN individualmente
(requiere lectura manual de posts representativos).

## 7. Convergencia de señales por actor

Aquí hacemos lo más valioso del análisis multi-técnica: ver si **diferentes técnicas
independientes señalan a los mismos actores** como relevantes.

Si un usuario aparece como:
- Top betweenness en la red (notebook 02)
- Top broker en PMs (notebook 02)
- Entidades NER muy específicas (notebook 03)
- Cluster ideológico diferenciado (notebook 03)

...entonces la evidencia de su importancia es mucho más sólida que si solo aparece en una técnica.

In [8]:
# Tabla de convergencia: una fila por usuario, una columna por señal
# Normalizar cada señal a escala 0-1 para poder comparar

def normalize(series):
    """Normalizar una serie a escala 0-1. Maneja NaN sin problema."""
    mn, mx = series.min(), series.max()
    if mx == mn: return series * 0  # Todos iguales → todos cero
    return (series - mn) / (mx - mn)

if len(actors_table) > 0:
    convergence = actors_table.copy()

    # Normalizar señales disponibles
    signals_to_norm = []
    if 'post_count' in convergence.columns:
        convergence['score_volume'] = normalize(convergence['post_count'].fillna(0))
        signals_to_norm.append('score_volume')
    if 'betweenness_public' in convergence.columns:
        convergence['score_btw_pub'] = normalize(convergence['betweenness_public'].fillna(0))
        signals_to_norm.append('score_btw_pub')
    if 'betweenness_pm' in convergence.columns:
        convergence['score_btw_pm'] = normalize(convergence['betweenness_pm'].fillna(0))
        signals_to_norm.append('score_btw_pm')

    if signals_to_norm:
        # Score combinado: promedio de las señales normalizadas disponibles
        convergence['combined_score'] = convergence[signals_to_norm].mean(axis=1)
        top_convergence = convergence.nlargest(20, 'combined_score')

        # Gráfica de radar / heatmap de señales
        disp_cols = ['username'] + signals_to_norm + ['combined_score']
        disp_cols = [c for c in disp_cols if c in top_convergence.columns]

        # Heatmap de señales normalizadas
        heat_df = top_convergence[signals_to_norm].fillna(0)
        heat_df.index = top_convergence['username'].fillna(top_convergence['userid']).tolist()

        col_labels = {
            'score_volume':  'Volumen posts',
            'score_btw_pub': 'Betweenness público',
            'score_btw_pm':  'Betweenness PMs',
        }

        fig = go.Figure(go.Heatmap(
            z=heat_df.values,
            x=[col_labels.get(c, c) for c in heat_df.columns],
            y=heat_df.index.tolist(),
            colorscale='YlOrRd',
            hovertemplate='%{y}<br>%{x}: %{z:.3f}<extra></extra>',
            colorbar=dict(title='Score<br>normalizado'),
        ))
        fig.update_layout(
            title='Convergencia de señales por actor — top 20 (0=mínimo, 1=máximo)',
            template='plotly_dark',
            height=max(400, 25 * len(heat_df)),
            margin=dict(l=160, r=20, t=60, b=40),
        )
        fig.show()

        print('\nScore combinado de los 10 actores con mayor convergencia de señales:')
        print(top_convergence[disp_cols].head(10).to_string(index=False))
else:
    print('No hay suficientes datos para calcular convergencia.')


Score combinado de los 10 actores con mayor convergencia de señales:
         username  score_volume  score_btw_pub  score_btw_pm  combined_score
Александр Славрос      1.000000       0.327229      1.000000        0.775743
         MOONLORD      0.362909       1.000000      0.001535        0.454815
     Daddy Terror      0.445794       0.065691      0.541990        0.351158
             Odin      0.208519       0.259322      0.309341        0.259061
         Змајевит      0.463171       0.250088      0.045408        0.252889
       Myrrysmies      0.514993       0.069517      0.157612        0.247374
           Aquila      0.347993       0.195533      0.161231        0.234919
           Spöket      0.518530       0.070226      0.073478        0.220745
  Moorish Fascist      0.431647       0.212508      0.012454        0.218870
     Clive Bissel      0.445333       0.070368      0.092010        0.202570


## 8. Tabla resumen de actores clave

Esta es la tabla de síntesis final. Integra todas las señales disponibles para cada
actor clave en un formato que puede entenderse sin conocimientos técnicos.

In [9]:
# Tabla resumen ejecutiva
if len(actors_table) > 0:
    summary_table = actors_table.nlargest(15, 'post_count').copy()

    # Columnas disponibles para mostrar
    show_cols = ['username', 'post_count']
    col_labels = {'username': 'Usuario', 'post_count': 'Posts'}

    optional_cols = [
        ('betweenness_public', 'Betweenness público'),
        ('betweenness_pm',     'Betweenness PMs'),
        ('hdbscan_cluster',    'Cluster semántico'),
        ('role',               'Rol (LLM)'),
        ('radicalization_level', 'Radicalización'),
        ('min_delta',          'Min Delta estilo'),
    ]
    for col, label in optional_cols:
        if col in summary_table.columns:
            show_cols.append(col)
            col_labels[col] = label

    display_df = summary_table[show_cols].rename(columns=col_labels)

    # Mostrar como tabla Plotly para mejor presentación
    fig = go.Figure(go.Table(
        header=dict(
            values=list(display_df.columns),
            fill_color='#2C2C2C',
            font=dict(color='white', size=11),
            align='left',
        ),
        cells=dict(
            values=[display_df[col].fillna('—').astype(str) for col in display_df.columns],
            fill_color='#1C1C1C',
            font=dict(color='white', size=10),
            align='left',
        )
    ))
    fig.update_layout(
        title='Resumen de actores clave — IronMarch',
        template='plotly_dark', height=500,
    )
    fig.show()
else:
    print('Sin datos suficientes para tabla resumen.')

## 9. Conclusiones

### ¿Qué respondimos?

🟢 **P1 — Actores clave**: identificamos una élite de ~5% de usuarios que genera la mayoría
del contenido. El rol de "líder ideológico" (alto betweenness público + vocabulario
filosófico) está respaldado por la centralidad de red del notebook 02. El perfilado LLM
(notebook 03, 32 actores) añade una segunda señal, complementaria y no siempre coincidente:
`Александр Славрос` (cuenta del nombre real del fundador) sale como `moderator`, mientras
`MOONLORD` (el alias con mayor betweenness público) sale solo como `active_member` — el
perfilado por contenido no sustituye a la centralidad estructural, ambas miden cosas
distintas.

🟡 **P2 — Sockpuppets**: el par con menor Delta de todo el corpus es `Rintrah` + `thecolonel`
(0.2914); en 2º lugar, `Александр Славрос` + `Daddy Terror` (0.2977) — conecta la cuenta del
fundador con el operador de PMs. El cruce con cluster semántico, similitud horaria y contacto
en PMs (sección siguiente) da 15 pares con 3 de 4 señales coincidentes, ninguno con las 4.
La estilometría no es prueba conclusiva en un corpus homogéneo.

🟢 **P3 — Eventos externos**: los atentados de París (nov 2015) generaron el único spike de
activación confirmado (z=2.34). Trump 2016 y Charlottesville 2017 dispararon la mención del
tema pero no la activación — hablar de algo no es lo mismo que activarse por ello.

🟢 **P4 — Comunidades**: comunidades de red (Leiden, notebook 02) y clusters semánticos
(HDBSCAN, notebook 03) son señales independientes (Adjusted Rand Index = 0.0173) — con quién
compartes hilo y de qué hablas no son la misma cosa en este corpus.

### ¿Qué queda abierto?

- **Verificación de identidades**: las identidades propuestas (sockpuppets, líderes)
  requieren corroboración con fuentes externas (investigaciones judiciales previas, OSINT) —
  no son un hallazgo de este análisis de datos.

- **Análisis de contenido de PMs**: tenemos 55K mensajes privados casi sin explorar.
  Un análisis específico de PMs podría revelar coordinación privada y operaciones que
  no eran visibles en el foro público.

- **Rastreo de actores fuera del foro**: algunos usuarios con emails conocidos
  (especialmente .edu) podrían rastrearse en otras plataformas.

- **Análisis en alemán y ruso**: ~15% de los posts no son en inglés y no fueron
  analizados con el pipeline semántico actual.

### ¿Por qué este dataset es excepcional?

La mayoría de leaks de foros solo contienen posts públicos. IronMarch incluye **mensajes privados**,
que son la capa de coordinación que normalmente queda invisible. La existencia de **ground truth**
(miembros identificados en investigaciones posteriores, el fundador doxxeado) permite validar
el pipeline computacional de una manera que rara vez es posible.

## 10. Exportación del informe

### Opción A: Exportar como HTML interactivo

El comando de abajo convierte el notebook en un archivo HTML que incluye
todas las gráficas interactivas de Plotly. Puede abrirse en cualquier navegador
sin necesidad de Python o Jupyter.

```bash
jupyter nbconvert --to html 04_sintesis_informe.ipynb --output informe_ironmarch.html
```

### Opción B: Exportar como PDF

Requiere tener `pandoc` y una instalación de LaTeX instalada.
Las gráficas interactivas se convierten en imágenes estáticas.

```bash
jupyter nbconvert --to pdf 04_sintesis_informe.ipynb --output informe_ironmarch.pdf
```

### Opción C: Exportar como HTML ejecutable completo

Este formato incluye las salidas ya calculadas sin necesidad de re-ejecutar el notebook:

```bash
jupyter nbconvert --to html --execute 04_sintesis_informe.ipynb
```

In [10]:
import subprocess
import sys

# Exportar este notebook como HTML
# Ejecutar esta celda genera el informe en HTML directamente desde Python
output_name = IM_RESULTS / 'informe_ironmarch.html'

result = subprocess.run(
    [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'html',
     '04_sintesis_informe.ipynb',
     '--output', str(output_name)],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f'Informe HTML generado: {output_name}')
    print(f'Tamaño: {output_name.stat().st_size / 1024:.0f} KB')
else:
    print('Error al generar HTML:')
    print(result.stderr)

Informe HTML generado: /home/miguel/csbc26/results/ironmarch/informe_ironmarch.html
Tamaño: 420 KB


In [11]:
# Guardar tabla resumen de actores como CSV (para compartir con no-técnicos)
if len(actors_table) > 0:
    csv_path = IM_RESULTS / 'actores_clave_resumen.csv'
    actors_table.nlargest(50, 'post_count').to_csv(csv_path, index=False)
    print(f'CSV guardado: {csv_path}')

print('\n=== ANÁLISIS COMPLETO — IronMarch ===')
print('Notebooks ejecutados:')
print('  00_reconocimiento.ipynb     — Exploración y preguntas de investigación')
print('  01_ingenieria_datos.ipynb   — Limpieza y preparación')
print('  02_analisis_estructural.ipynb — Red, temporal, TF-IDF')
print('  03_analisis_semantico.ipynb — Embeddings, BERTopic, NER, Delta')
print('  04_sintesis_informe.ipynb   — Integración y conclusiones')
print()
print('Archivos generados en results/ironmarch/:')
for f in sorted(IM_RESULTS.iterdir()):
    if not f.name.startswith('.'):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<40} {size_kb:>8.0f} KB')

CSV guardado: /home/miguel/csbc26/results/ironmarch/actores_clave_resumen.csv

=== ANÁLISIS COMPLETO — IronMarch ===
Notebooks ejecutados:
  00_reconocimiento.ipynb     — Exploración y preguntas de investigación
  01_ingenieria_datos.ipynb   — Limpieza y preparación
  02_analisis_estructural.ipynb — Red, temporal, TF-IDF
  03_analisis_semantico.ipynb — Embeddings, BERTopic, NER, Delta
  04_sintesis_informe.ipynb   — Integración y conclusiones

Archivos generados en results/ironmarch/:
  actor_profiles.parquet                         21 KB
  actores_clave_resumen.csv                       6 KB
  bertopic_topic_info.parquet                    19 KB
  bertopic_topics.parquet                     31411 KB
  corpus_users_clean.parquet                  13966 KB
  delta_pairs.parquet                            99 KB
  forums_clean.parquet                            6 KB
  informe_ironmarch.html                        420 KB
  ner_comparison.parquet                         23 KB
  ner_results.p